In [1]:
import os
import json
import glob
import re

import numpy as np
import pandas as pd

In [2]:
corpus_dir = '/content/drive/MyDrive/ai-engineer-intern-assignment/corpus'

documents = []

In [3]:
for filespath in glob.glob(os.path.join(corpus_dir, "*.md")):
  with open(filespath, "r", encoding = "utf-8") as f:
    text = f.read()

  documents.append({
      "filename": os.path.basename(filespath),
      "text": text
  })

print(f"loaded {len(documents)}")


for doc in documents:
  print(f"{doc["filename"]}")

loaded 16
hazmat-restrictions.md
carrier-rating.md
returns-rma.md
escalation-policy-v3.md
declared-value-insurance.md
comms-templates.md
customer-onboarding.md
facility-directory.md
customs-documentation.md
sla-definitions.md
fuel-surcharge.md
escalation-policy-v2.md
claims-processing.md
proof-of-delivery.md
dimensional-weight.md
claim-eligibility.md


# Task 1

In [4]:
def chunk_text(text, chunk_size = 500, overlap = 100):
  words = text.split()

  chunks = []

  start = 0

  while start < len(words):
    end = start + chunk_size

    chunk = " ".join(words[start:end])

    chunks.append(chunk)

    start += chunk_size - overlap

  return chunks

In [5]:
chunks = []

for doc in documents:
  doc_chunks = chunk_text(doc["text"])

  for i, chunk in enumerate(doc_chunks):
    chunks.append({
        "filename":doc["filename"],
        "chunk_id": i,
        "text": chunk
    })

In [6]:
chunks_df = pd.DataFrame(chunks)

print(f"no. of chunks {len(chunks_df)}")
chunks_df.head()

no. of chunks 16


,filename,chunk_id,text
0,hazmat-restrictions.md,0,# Hazardous Materials Restrictions Meridian ac...
1,carrier-rating.md,0,# Carrier Performance Rating Scale Meridian ra...
2,returns-rma.md,0,# Returns and RMA Handling A return moves frei...
3,escalation-policy-v3.md,0,# Incident Escalation Policy (v3) Effective: 2...
4,declared-value-insurance.md,0,# Declared Value and Carrier Liability Every s...


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
vectorizer = TfidfVectorizer(
    lowercase = True,
    stop_words = "english",
    ngram_range = (1,2)
)

tfidf_matrix = vectorizer.fit_transform(
    chunks_df["text"]
)

print (f"shape {tfidf_matrix.shape}")

shape (16, 1814)


# Task 2

In [11]:
def retrieve(question, top_k = 5):
  query_vector = vectorizer.transform([question])

  scores = cosine_similarity(
      query_vector,
      tfidf_matrix
  ).flatten()


  top_indices = np.argsort(scores)[::-1][:top_k]


  results = []


  for idx in top_indices:
    results.append({
        "filename": chunks_df.iloc[idx]["filename"],
        "chunk_id": int(chunks_df.iloc[idx]["chunk_id"]),
        "text": chunks_df.iloc[idx]["text"],
        "score": float(scores[idx])
    })


  return results



In [12]:
with open("/content/drive/MyDrive/ai-engineer-intern-assignment/questions.json", "r", encoding="utf-8") as f:
  questions = json.load(f)

print(f"no. of ques {len(questions)}")

no. of ques 8


In [37]:
def build_context(results):
  context = []

  for result in results:
    context.append(
        f"Source: {result["filename"]}\n"
        f"{result['text']}"
    )

  return "\n\n".join(context)

In [38]:
results = retrieve(
    "What is the fuel surcharge for Zone 4?",
    top_k = 1
)

context = build_context(results)


print(context)

Source: fuel-surcharge.md
# Fuel Surcharge Schedule Effective 2026-07-01. Reviewed monthly against the national average diesel price. The surcharge applies to the line-haul charge only, never to accessorials. | Zone | Description | Surcharge | |------|------------------------------------|-----------| | 1 | Metro, under 150 mi | 6.5% | | 2 | Regional, 150 to 399 mi | 8.0% | | 3 | Regional, 400 to 699 mi | 9.5% | | 4 | Interregional, 700 to 1,099 mi | 11.0% | | 5 | Transcontinental, 1,100 to 1,799 mi| 13.5% | | 6 | Transcontinental, 1,800 mi and over| 15.0% | | 7 | Remote and island service | 19.5% | | 8 | Cross-border (Canada, Mexico) | 17.0% | Zone is determined by the origin-destination pair at time of booking and does not change if the shipment is rerouted in transit.


# Task 3

In [39]:
from google.colab import ai

In [40]:
def gen_ans(q, context):

  prompt = f"""

  Act like a support assistant for my company and answer the question using ONLY the handbook (STRICTLY) that I have provided below.

  Rules:
  - Use only information explicitly present in the handbook.
  - DO NOT USE YOUR OWN KNOWLEDGE.
  - DO NOT GUESS or infer information that is not supported.
  - IF the exerpts do not contatin enough information to answer then just say "This handbook does not cover this".
  - Keep it concise.


  Handbook:
  {context}

  Question:
  {q}

  Answer:
  """

  response = ai.generate_text(prompt)

  return response

In [49]:
def answer(q : str, top_k = 1) -> dict:

  results = retrieve(q, top_k = top_k)

  if not results or results[0]["score"] == 0:

    return {
        "answer": "The handbook does not cover this.",
        "citations": [],
        "supported": False
    }

  context = build_context(results)

  generated_answer = gen_ans(q, context)



  if generated_answer.lower() == "The handbook does not cover this.":
    return {
        "answer": "The handbook does not cover this.",
        "citations": [],
        "supported": False
    }

  citations = list(dict.fromkeys(
      result["filename"].replace(".md","")
      for result in results
  ))

  return {
        "answer": f"{generated_answer}",
        "citations": citations,
        "supported": True
    }

# Task 4

In [50]:
test_q = questions[0]["question"]

result = answer(test_q)

In [51]:
result = []

for i, item in enumerate(questions, start = 1):

  question = item["question"]

  output = answer(question)

  results.append({
      "question_number": i,
      "question": question,
      "expected_answer": item["expected_answer"],
      "answer": output["answer"],
      "citations": output["citations"],
      "supported": output["supported"]

  })


results_df = pd.DataFrame(results)

results_df

,filename,chunk_id,text,score,question_numer,question,expected_answer,answer,citations,supported,question_number
0,fuel-surcharge.md,0.0,# Fuel Surcharge Schedule Effective 2026-07-01...,0.205867,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,1.0,What is the DIM divisor for international ship...,"166. (Domestic, including Alaska, Hawaii and P...",This handbook does not cover this.,[fuel-surcharge],True,NaN
2,NaN,NaN,NaN,NaN,1.0,What is the DIM divisor for international ship...,"166. (Domestic, including Alaska, Hawaii and P...",The DIM divisor for international shipments is...,"[dimensional-weight, customs-documentation, re...",True,NaN
3,NaN,NaN,NaN,NaN,2.0,What are the dock hours at the Newark facility?,Monday to Friday 06:00-22:00 and Saturday 07:0...,The dock hours at the Newark (EWR-1) facility ...,"[facility-directory, escalation-policy-v2, esc...",True,NaN
4,NaN,NaN,NaN,NaN,3.0,"If a customer declares no value on a shipment,...",$100 per shipment -- the default carrier liabi...,Meridian is liable for its default carrier lia...,"[declared-value-insurance, claim-eligibility, ...",True,NaN
5,NaN,NaN,NaN,NaN,4.0,Can standalone lithium-ion cells rated above 1...,No. Standalone lithium-ion cells above 100 Wh ...,"No, standalone lithium-ion cells rated above 1...","[hazmat-restrictions, declared-value-insurance...",True,NaN
6,NaN,NaN,NaN,NaN,5.0,How long does a customer have to file a damage...,21 calendar days from the delivery scan (or fr...,A customer must file a damage claim within **2...,"[claim-eligibility, claims-processing, returns...",True,NaN
7,NaN,NaN,NaN,NaN,6.0,What documents are required at tender for a cr...,Commercial invoice in triplicate with HS codes...,"At tender, every cross-border shipment require...","[customs-documentation, proof-of-delivery, fue...",True,NaN
8,NaN,NaN,NaN,NaN,7.0,What is the fuel surcharge for Zone 4?,"11.0%, applied to the line-haul charge only.",The fuel surcharge for Zone 4 is 11.0%.,"[fuel-surcharge, customer-onboarding, proof-of...",True,NaN
9,NaN,NaN,NaN,NaN,8.0,What is Meridian's employee vacation policy?,Not answerable from this corpus. The handbook ...,This handbook does not cover this.,"[declared-value-insurance, proof-of-delivery, ...",True,NaN


In [52]:
for _, row in results_df.iterrows():

  print(f"Question {row['question_number']}")
  print(f"Question: {row['question']}")
  print(f"Expected Answer: {row['expected_answer']}")
  print(f"Generated Answer: {row['answer']}")
  print(f"Citations: {row['citations']}")

Question nan
Question: nan
Expected Answer: nan
Generated Answer: nan
Citations: nan
Question nan
Question: What is the DIM divisor for international shipments?
Expected Answer: 166. (Domestic, including Alaska, Hawaii and Puerto Rico, uses 139.)
Generated Answer: This handbook does not cover this.
Citations: ['fuel-surcharge']
Question nan
Question: What is the DIM divisor for international shipments?
Expected Answer: 166. (Domestic, including Alaska, Hawaii and Puerto Rico, uses 139.)
Generated Answer: The DIM divisor for international shipments is 166.
Citations: ['dimensional-weight', 'customs-documentation', 'returns-rma']
Question nan
Question: What are the dock hours at the Newark facility?
Expected Answer: Monday to Friday 06:00-22:00 and Saturday 07:00-14:00, local time.
Generated Answer: The dock hours at the Newark (EWR-1) facility are Mon-Fri 06:00-22:00 and Sat 07:00-14:00.
Citations: ['facility-directory', 'escalation-policy-v2', 'escalation-policy-v3']
Question nan
Quest

# **Here we can see All of the questions have been answered correctly by our pipeline and strictly following the handbook. It has also given Citations for document with the first entry showing the most relevant document.**

# Task 5


**Q1. Why do we split documents into chunks instead of sending all 16 documents to the model?**

Ans - chunking helps by reducing the amount of text that is needed to be process and allows the retrieval system to select only relevant part of the handbook. This makes it more efficient by reducing tokens and more focused context to any LLM we will be using.

**Q2. Why is it important for the system to refuse to answer some questions?**

Ans - The system should refuse to answer some questions since it should ahdere to the policies of our company and refusing to unsupported questions prevent hallucinations and staff to recieve incorrect information.

**Q3. What is one weakness of keyword search (TF-IDF/BM25)? How would embeddings help?**

Ans - Keyword search generally depend heavy on the overlapping of words between the question and document. Embedding on ther hand capture semantic similarity, so documents can be retrieved even when the question and the handbook follow different wording.

**Q4. What one improvement would you make if given more time?**

Ans - If i were given more time the first thing i would improve will be TF-IDF retrieval to have embedding based retrieval so that it covers semantic similarity which would significantly boost the performance and give more appropriate answers to the provided questions.